# Capture temperature over time using BMP581 chip and plot it.

First establish communication to the I2C dongle, create link to the BMP581 chip on the I2C bus and obtain address map for this chip.

In [ ]:
import time
import matplotlib.pyplot as plt
from rgscore import AddrMap, RLinkI2C, read, write, S
from i2c_drivers import I2CMasterI2CDriver

In [ ]:
# First create instance of I2CMaster; depending on the dongle that you have that could be I2CMasterMCP2221, I2CMasterFT232H or I2CMasterI2CDriver
i2c_master = I2CMasterI2CDriver("/dev/ttyUSB0") # COM port might different; you will need to find on which port the dongle is connected

# Now create link to the BMP581 device on the I2C bus. In this example case it is at I2C address 0x47.
link = RLinkI2C(i2c_master, device_address = 0x47)

# Create instance of address map for this chip.
addrMap = AddrMap.new_instance_from_url("https://raw.githubusercontent.com/Brainless-Software/rgscore/refs/heads/master/devices/AddrMap_BMP581.json", link)

Now obtain references to `regList` (simplified registers model) and `dut`, live _Device Under Test_ object.

In [ ]:
regList = addrMap.regList
dut = addrMap.dut

Now we are ready to start talking to BMP581.

First we are going to issue soft reset.

In [ ]:
dut.CMD.cmd = regList.CMD.cmd.soft_reset
write(dut.CMD)

Take one time temperature measurment

In [ ]:
read(dut.ODR_CONFIG)

In [ ]:
# setting ODR_CONFIG.pwr_mode to `Forced` triggers one time temperature measurement. 
dut.ODR_CONFIG.pwr_mode = regList.ODR_CONFIG.pwr_mode.Forced
write(dut.ODR_CONFIG)

# measurement does take some time; hence the delay
time.sleep(0.05) 

# Now read back three registers that hold temperature data
read(dut.TEMP_DATA_LSB, dut.TEMP_DATA_XLSB, dut.TEMP_DATA_MSB)

# Compbine them into one large signed fixed point number S24.16
temp_c = S(16, [dut.TEMP_DATA_MSB.temp_23_16, dut.TEMP_DATA_LSB.temp_15_8, dut.TEMP_DATA_XLSB.temp_7_0])
print(f"Temperature {temp_c} C")

Now lets run it in a loop for a minute and plot results.

In [ ]:
temp_measurements = []

start_at = time.time()
while time.time() - start_at < 60:
    dut.ODR_CONFIG.pwr_mode = regList.ODR_CONFIG.pwr_mode.Forced
    write(dut.ODR_CONFIG)
    time.sleep(1)
    read(dut.TEMP_DATA_LSB, dut.TEMP_DATA_XLSB, dut.TEMP_DATA_MSB)
    temp_c = S(16, [dut.TEMP_DATA_MSB.temp_23_16, dut.TEMP_DATA_LSB.temp_15_8, dut.TEMP_DATA_XLSB.temp_7_0])
    temp_measurements.append(temp_c)

In [ ]:
plt.plot(temp_measurements)